<a href="https://colab.research.google.com/github/Opm-dlta/Bigdata/blob/main/nurse_finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import json

file_paths = [r"/content/sample_data/test_unsloth.jsonl", r"/content/sample_data/valid_unsloth.jsonl", r"/content/sample_data/test_unsloth.jsonl", r"/content/sample_data/test_unsloth.jsonl"]
file = []
for file_path in file_paths:
    with open(file_path, "r") as f:
        for line in f:
            file.append(json.loads(line))

print(f"Loaded {len(file)} records from all files.")
print(file[1])

Loaded 576 records from all files.
{'conversations': [{'from': 'system', 'value': 'You are Pyra, a kind and caring nurse. Provide gentle and accurate medical advice to help your patient feel safe.'}, {'from': 'human', 'value': 'A 75-year-old man presents with the following symptoms: runny nose, sneezing, mild cough, sore throat. The symptoms have persisted for 6 days, and the patient has a temperature of 37.5°C. What is the recommended treatment?'}, {'from': 'assistant', 'value': '1. Rest and limit activity for 2–3 days.\n2. Drink ≥2 L fluids daily (water, broth, tea).\n3. Paracetamol 500 mg or ibuprofen 200–400 mg every 6 h for fever/aches.\n4. Saline nasal spray 3 times/day; lozenges or cough syrup PRN.\n5. If ≤48 h since symptom onset, consider antiviral per guidelines.'}]}


In [2]:
!pip uninstall -y unsloth peft

!pip install unsloth trl peft accelerate bitsandbytes

Found existing installation: peft 0.16.0
Uninstalling peft-0.16.0:
  Successfully uninstalled peft-0.16.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 2.5 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.3/299.3 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.2/376.2 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.5/166.5 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.1/117.1 MB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 821.2/821.2 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━

In [3]:
# For GPU check
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

CUDA available: True
GPU: Tesla T4


In [4]:
from datasets import load_dataset
import json

def format_conversation(example):
    formatted_text = ""
    for turn in example['conversations']:
        role = turn['from']
        content = turn['value']
        # Customize the formatting based on your model's expected prompt structure
        if role == 'system':
            formatted_text += f"<|system|>\n{content}<|endoftext|>\n"
        elif role == 'human':
            formatted_text += f"<|user|>\n{content}<|endoftext|>\n"
        elif role == 'assistant':
            formatted_text += f"<|assistant|>\n{content}<|endoftext|>\n"
    return {"text": formatted_text, "conversations": example['conversations']}

# Paths to your data
train_path = "/content/sample_data/train_unsloth.jsonl"
valid_path = "/content/sample_data/valid_unsloth.jsonl"
test_path = "/content/sample_data/test_unsloth.jsonl"

def get_dataset(path):
    return load_dataset("json", data_files=path, split="train")

train_dataset = get_dataset(train_path)
valid_dataset = get_dataset(valid_path)
test_dataset = get_dataset(test_path)


# Apply the formatting function to the datasets
train_dataset = train_dataset.map(format_conversation)
valid_dataset = valid_dataset.map(format_conversation)
test_dataset = test_dataset.map(format_conversation)


print("Formatted training dataset example:")
print(train_dataset[0]["text"])

print("\nFormatted validation dataset example:")
print(valid_dataset[0]["text"])

print("\nFormatted test dataset example:")
print(test_dataset[0]["text"])

output_dir = "/content/sample_data/"
train_output_path = output_dir + "IDk_train_unsloth.jsonl"
valid_output_path = output_dir + "IDk_valid_unsloth.jsonl"
test_output_path = output_dir + "IDk_test_unsloth.jsonl"

train_dataset.to_json(train_output_path, orient="records", lines=True, force_ascii=False)
valid_dataset.to_json(valid_output_path, orient="records", lines=True, force_ascii=False)
test_dataset.to_json(test_output_path, orient="records", lines=True, force_ascii=False)

print(f"\nSaved formatted training data to {train_output_path}")
print(f"Saved formatted validation data to {valid_output_path}")
print(f"Saved formatted test data to {test_output_path}")

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/96 [00:00<?, ? examples/s]

Map:   0%|          | 0/288 [00:00<?, ? examples/s]

...

Formatted training dataset example:
<|system|>
You are Pyra, a kind and caring nurse. Provide gentle and accurate medical advice to help your patient feel safe.<|endoftext|>
<|user|>
A 64-year-old man presents with a superficial (1st°) burn covering 2% of body surface area, caused by sunlamp overexposure 6.7 hours ago. The patient reports moderate pain (6/10). What is the recommended treatment?<|endoftext|>
<|assistant|>
1. Cool under running water (15–20 °C) for 10–20 min.
2. Gently pat dry and apply 1 % silver sulfadiazine or aloe vera gel.
3. Cover with non‐adhesive sterile dressing and change daily.
4. Ibuprofen 200–400 mg every 6–8 h for pain.
5. Watch for infection (increased redness, swelling, pus).<|endoftext|>


Formatted validation dataset example:
<|system|>
You are Pyra, a kind and caring nurse. Provide gentle and accurate medical advice to help your patient feel safe.<|endoftext|>
<|user|>
A 15-year-old man presents with a partial‐thickness (2nd°) blistered burn covering 8

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]


Saved formatted training data to /content/sample_data/IDk_train_unsloth.jsonl
Saved formatted validation data to /content/sample_data/IDk_valid_unsloth.jsonl
Saved formatted test data to /content/sample_data/IDk_test_unsloth.jsonl


In [5]:
from unsloth import FastLanguageModel
from transformers import TrainingArguments
from trl import SFTTrainer
from datasets import load_dataset
import torch

BASE_MODEL = "unsloth/llama-2-7b-bnb-4bit"
max_seq_length = 2048
dtype = None

train_path = "/content/sample_data/IDk_train_unsloth.jsonl"
valid_path = "/content/sample_data/IDk_valid_unsloth.jsonl"
test_path = "/content/sample_data/IDk_test_unsloth.jsonl"

def get_dataset(path):
    return load_dataset("json", data_files=path, split="train")

train_dataset = get_dataset(train_path)
valid_dataset = get_dataset(valid_path)

def format_conversation(example):
    formatted_text = ""
    for turn in example['conversations']:
        role = turn['from']
        content = turn['value']
        if role == 'system':
            formatted_text += f"<|system|>\n{content}<|endoftext|>\n"
        elif role == 'human':
            formatted_text += f"<|user|>\n{content}<|endoftext|>\n"
        elif role == 'assistant':
            formatted_text += f"<|assistant|>\n{content}<|endoftext|>\n"
    return {"text": formatted_text}


train_dataset = train_dataset.map(format_conversation, remove_columns=["conversations"])
valid_dataset = valid_dataset.map(format_conversation, remove_columns=["conversations"])


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=True,
    device_map="auto"
)

model = FastLanguageModel.get_peft_model(
    model,
    r=64,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=128,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)


training_args = TrainingArguments(
    output_dir="results/pyra-llama2-finetune",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    num_train_epochs=3,
    eval_strategy="steps",
    eval_steps=100,
    save_steps=100,
    logging_steps=50,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    report_to="none",
    save_total_limit=2,
    dataloader_pin_memory=False,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    args=training_args,
)

trainer.train()

trainer.save_model("results/pyra-llama2-finetune/final_model")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/96 [00:00<?, ? examples/s]

Map:   0%|          | 0/288 [00:00<?, ? examples/s]

...

Unsloth: Tokenizing ["text"]:   0%|          | 0/96 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"]:   0%|          | 0/288 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 96 | Num Epochs = 3 | Total steps = 36
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 159,907,840 of 6,898,323,456 (2.32% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss


In [ ]:
from unsloth import FastLanguageModel
import torch
import json
import os

finetuned_model_path = "results/pyra-llama2-finetune/final_model"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=finetuned_model_path,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
    device_map="auto"
)

def handle_conversation():
    BLUE = "\033[94m"
    RED = "\033[91m"
    RESET = "\033[0m"
    context = ""
    print("Hi am Pyra ready to answer your questions. about your health. When you want to end the conversation Type 'exit' to quit.")
    while True:
        user_input = input(f"{BLUE}User: {RESET}")
        if user_input.lower() == "exit":
            print("Goodbye!")
            break

        current_prompt = f"<|system|>\nYou are Pyra, a kind and caring nurse. Provide gentle and accurate medical advice to help your patient feel safe.<|endoftext|>\n{context}<|user|>\n{user_input}<|endoftext|>\n<|assistant|>"

        inputs = tokenizer(current_prompt, return_tensors="pt").to("cuda")

        outputs = model.generate(**inputs, max_new_tokens=256, use_cache=True)

        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=False)
        assistant_response_start = generated_text.rfind("<|assistant|>")
        if assistant_response_start != -1:
            assistant_response = generated_text[assistant_response_start + len("<|assistant|>"):].strip()
            assistant_response = assistant_response.split("<|endoftext|>")[0].strip()
        else:
            assistant_response = generated_text.strip()

        print(f"{RED}Pyra: {assistant_response}{RESET}")

        context += f"<|user|>\n{user_input}<|endoftext|>\n<|assistant|>\n{assistant_response}<|endoftext|>\n"

if __name__ == "__main__":
    handle_conversation()